# Topic: Safe Eval/Exec Alternatives - Runtime code execution risks, eval exploits, and safe AST literal evaluations
 
## 1. THE DANGER OF eval() AND exec()
- **`eval(string)`**: Evaluates a Python expression string dynamically at runtime and returns the result.
- **`exec(string)`**: Executes a block of Python code dynamically.
- **The Security Risk**: If an untrusted user input is passed directly to `eval` or `exec`, it enables **Remote Code Execution (RCE)**.
  - An attacker can inject imports to run system commands:
    `__import__('os').system('whoami')`
  - Even attempts to restrict scope by passing custom globals dictionaries (`eval(input_str, {"__builtins__": None})`) can be bypassed by traversing python's object inheritance graph to find access to classes.
 
## 2. THE SAFE ALTERNATIVE: ast.literal_eval()
- **`ast.literal_eval(string)`**:
  - Provided by Python's built-in **`ast`** (Abstract Syntax Tree) library.
  - Safely parses and evaluates string representations of Python literal structures: Strings, Numbers, Tuples, Lists, Dicts, Booleans, and None.
  - **How it works**: It inspects the syntax tree node-by-node and throws a `ValueError` if the node is not a primitive literal. It **never compiles or executes active code**, blocking function calls, imports, and system commands completely.
 
---


In [ ]:
import ast

# 1. Simulating a user config parser using eval() (Vulnerable)
def vulnerable_parse_config(user_input_str):
    """Parses a user dictionary string using eval (DANGEROUS!)."""
    print(f"[VULNERABLE EVALUATING]: {user_input_str}")
    # Attempts to restrict globals, but is still vulnerable
    return eval(user_input_str, {"__builtins__": {}})

# Normal usage works
config_normal = "{'theme': 'dark', 'version': 1.0}"
print(f"Normal parse result: {vulnerable_parse_config(config_normal)}")



In [ ]:
print("\n--- Command Injection Attack on eval() ---")
# Attacker bypasses the empty __builtins__ by traversing object hierarchies
# to access the sys or os modules.
# Exploit payload that reconstructs os.system:
exploit_payload = "[c for c in ().__class__.__base__.__subclasses__() if c.__name__ == 'BuiltinImporter'][0]().load_module('os').system('echo VULNERABILITY_EXPLOITED')"

try:
    vulnerable_parse_config(exploit_payload)
except Exception as e:
    print(f"Execution Error: {e}")
# Expected: Executes 'echo VULNERABILITY_EXPLOITED' directly in the host OS!



In [ ]:
# 2. Secure parsing using ast.literal_eval
def secure_parse_config(user_input_str):
    """Parses a user dictionary string safely using ast.literal_eval."""
    print(f"[SECURE PARSING]: {user_input_str}")
    try:
        # ast.literal_eval only parses static primitives, never executes operations
        return ast.literal_eval(user_input_str)
    except (ValueError, SyntaxError) as e:
        return f"Secure Block: Invalid configuration format! Details: {e}"

print("\n--- Secure Parse Validation ---")
# Normal input works
print(f"Normal parse result: {secure_parse_config(config_normal)}")

# Attack payload is blocked
print(f"Attack parse result: {secure_parse_config(exploit_payload)}")
# Expected: Successfully blocks execution, throwing a ValueError!
